In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost lightgbm tqdm -q

clear_output()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
import warnings
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
# Task 1: Write your code here:
file_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(file_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
target_column = 'Delivery_Time'
df[target_column].dropna().hist(bins=30, edgecolor='black')

plt.title(f"Target Distribution ({target_column})")
plt.xlabel(target_column)
plt.ylabel("Frequency")
plt.grid(False)

In [ ]:
# Task 1: Write your code here:
df.drop('Order_ID', axis=1, inplace=True)

In [ ]:
# Task 2: Write your code here:
display(df.isnull().sum())

''' We cannot fill the nan values in the target cus it will not be logical to fill neither the mean nor the meadian which will effect our model learning '''
df.dropna(subset=target_column, inplace=True)

display(df.isnull().sum())

''' Since the dataset is small i will spare the nan values in the other columns with the appropriate values '''
object_nan = ['Weather', 'Traffic_Level', 'Time_of_Day']

for nan in object_nan:
  df[nan] = df[nan].fillna(df[nan].mode()[0])

display(df.isnull().sum())

df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean())

display(df.isnull().sum())

In [ ]:
# Task 3: Write your code here:
dups = int(df.duplicated().sum())
print('Number of duplicated rows = ', dups)
df.drop_duplicates(inplace=True)
dups_after = int(df.duplicated().sum())
print('Number of duplicated rows after deleting = ', dups_after)

In [ ]:
# Task 4: Write your code here:
categorical_columns = list(df.select_dtypes('object').columns)
categorical_columns

numerical_columns = list(df.select_dtypes(include = ['int64', 'float64']).columns) # Will be used later for scaling..

df = pd.get_dummies(df, columns=categorical_columns, drop_first=False)

In [ ]:
df = df.astype(float)
df

In [ ]:
# Task 5: Write your code here:
scaler = StandardScaler()

df[numerical_columns] = scaler.fit_transform(df[numerical_columns])
df

In [ ]:
# Task 6: Write your code here:

''' Based on the histrogram of the target column (Delivery_Time) it takes somewhat a normal curve neither right skewed nor left skewed '''

# from scipy import stats
# stats.

In [ ]:
# Task 1: Write your code here:
X = df.drop(target_column, axis=1)
y = df[target_column]

In [ ]:
# Task 2,3,4,5: Write your code here:
n_splits = 5

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

model = RandomForestRegressor(n_estimators=200)

mae_losses = []
y_preds = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  print(f"Training RandomForst...")

  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)

  mae = mean_absolute_error(y_test, y_pred)

  mae_losses.append(mae)

print('Average MAE loss:', np.mean(mae_losses))

In [ ]:
# Task 1: Write your code here:
RandomForest_model = model
RandomForest_importance = list(zip(X.columns, RandomForest_model.feature_importances_))
sorted_RandomForest_importance = sorted(RandomForest_importance, key=lambda x: abs(x[1]), reverse=True)

features, coefficients = zip(*sorted_RandomForest_importance)

plt.figure(figsize=(10, 6))
plt.barh(features, coefficients, color='darkred')
plt.xlabel('Coefficient Value')
plt.ylabel('Features')
plt.title('RandomForest Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure()
plt.hist(y_pred, bins=30, edgecolor='black')
plt.show()

In [ ]:
# Task Bonus: Write your code here: